# PHYS 690 — Lecture 20: PyTorch Flows for Ramsey Posterior Inference

Lectures 18 and 19 used `emcee` to sample and diagnose the posterior for a **single, fixed Ramsey dataset**. Today we keep exactly those simulated counts and infer four parameters, $\boldsymbol\theta=(\tau,C,\omega,\phi)$, as in Lecture 18's extension. We will use PyTorch and a ready-made flow to approximate the posterior **directly from its prior and binomial likelihood**, then compare the answer with `emcee`.

The extra frequency and phase parameters make the posterior more demanding than the earlier $(\tau,C)$ grid. The data and single-frequency physical model remain the same, so we can focus on how the inference method changes.

**Question for discussion:** If a flow produces thousands of parameter samples quickly after training, what computation must be included when deciding whether it was faster than `emcee`?

## Learning goals

By the end of class, you should be able to:

1. state what a PyTorch tensor, automatic differentiation, optimizer, and normalizing flow do in this inference problem;
2. express the Ramsey binomial log likelihood in PyTorch and check it against the earlier SciPy calculation;
3. explain the direct variational objective used to train a flow without first fitting it to MCMC samples;
4. compare flow and `emcee` posterior summaries and Ramsey predictions; and
5. identify when a flow approximation might miss important posterior structure.

Two coding activities ask you to complete the likelihood and the posterior comparison. The flow architecture and training loop are provided as worked code.

## Environment setup

Run these commands in a terminal at the **course repository root**, where `requirements.txt` is located. Use the virtual environment from Lecture 18. If `.venv` already exists, skip the creation command and the `requirements.txt` installation; activate it before installing the two new packages. (`torch` and `zuko` are not in the course `requirements.txt`.)

**macOS / Linux:**

```bash
python3 -m venv .venv                 # only if .venv does not exist
source .venv/bin/activate
python -m pip install -r requirements.txt  # only for a new .venv
python -m pip install torch zuko
python -c "import sys; print(sys.executable)"
```

**Windows using Git Bash:**

```bash
python -m venv .venv                  # only if .venv does not exist
source .venv/Scripts/activate
python -m pip install -r requirements.txt  # only for a new .venv
python -m pip install torch zuko
python -c "import sys; print(sys.executable)"
```

The printed Python path should be inside this repository's `.venv`. Activating the environment changes the **terminal** Python; it does not automatically change an already open notebook. In VS Code, click **Select Kernel** and choose the Python interpreter in this `.venv`; in JupyterLab, choose the corresponding kernel. Then **restart the kernel** and run the verification cell below. Its `Notebook Python` path should match the terminal environment. If it does not, select the kernel again before running the rest of the notebook. See [Lecture 18](lecture18_11_10_26_mcmc.ipynb) and the [course VS Code environment guide](../resources/vscode.md) for more detail.

In [ ]:
# Check which Python the notebook kernel is using before importing new packages.
import sys
print('Notebook Python:', sys.executable)

# Confirm that PyTorch and Zuko are installed in this kernel's environment.
import torch
import zuko
print('PyTorch:', torch.__version__, '| Zuko:', zuko.__version__)

## What PyTorch does in this lecture

[PyTorch](https://docs.pytorch.org/tutorials/beginner/basics/intro.html) is a Python library for numerical calculations with **tensors** (multidimensional arrays), automatic differentiation, and optimization. Its array operations resemble NumPy's, but PyTorch can record how a computed quantity depends on adjustable inputs. Calling `loss.backward()` then calculates derivatives through those operations. This extends the derivative ideas from Lecture 6: here the derivatives guide the optimization of an entire probability density, rather than a few fit parameters. We use the CPU; a GPU is not needed for this example.

| Concept | Role in the Ramsey calculation |
|---|---|
| **Tensor** | Stores numerical values. The observations remain in a pandas DataFrame for inspection and plots; we copy its columns to tensors for repeated likelihood evaluations. |
| **Automatic differentiation (autograd)** | Finds how a loss changes when trainable parameters change, provided the calculation uses differentiable PyTorch operations. |
| **Optimizer** | `torch.optim.Adam` uses those derivatives to update trainable parameters in small steps. |
| **Training** | Repeats the calculation, derivative, and update steps to reduce a chosen loss. |

### Worked example: derivatives and one parameter update

Suppose two measurements are $y_1=2$ and $y_2=1$. A small model predicts $m_1=a+b$ and $m_2=a-b$ from adjustable parameters $a$ and $b$. The sum of squared residuals is

$$
L(a,b)=(a+b-2)^2+(a-b-1)^2.
$$

Taking partial derivatives gives

$$
\frac{\partial L}{\partial a}=2(a+b-2)+2(a-b-1),\qquad
\frac{\partial L}{\partial b}=2(a+b-2)-2(a-b-1).
$$

At $(a,b)=(2,0)$, the loss is $1$ and the gradient is $(2,-2)$. Setting `requires_grad=True` lets PyTorch calculate those two derivatives with `backward()`. One gradient-descent step with learning rate $0.1$ updates $(a,b)$ to $(1.8,0.2)$ and lowers the loss to $0.36$. The same calculate-gradient-update pattern later trains the flow's many parameters, with Adam in place of this simple SGD optimizer.

In [ ]:
# Start with two trainable parameters and a simple optimizer.
a_demo = torch.tensor(2.0, requires_grad=True)
b_demo = torch.tensor(0.0, requires_grad=True)
optimizer_demo = torch.optim.SGD([a_demo, b_demo], lr=0.1)

def toy_loss():
    """Sum squared residuals for model predictions a+b and a-b."""
    residual_1 = a_demo + b_demo - 2.0
    residual_2 = a_demo - b_demo - 1.0
    return residual_1**2 + residual_2**2

# Differentiate the initial loss with respect to both parameters.
optimizer_demo.zero_grad()
initial_loss = toy_loss()
initial_loss.backward()
print(f'Initial loss: {initial_loss.item():.2f}')
print(f'Gradients: dL/da = {a_demo.grad.item():.1f}, '
      f'dL/db = {b_demo.grad.item():.1f}')

# Take one gradient-descent step and check that the loss decreases.
optimizer_demo.step()
print(f'Updated parameters: a = {a_demo.item():.1f}, '
      f'b = {b_demo.item():.1f}')
print(f'Updated loss: {toy_loss().item():.2f}')

## Why approximate a posterior with a normalizing flow?

The Ramsey counts $D$ and prior define the same target as in Lectures 18 and 19,

$$
p(\boldsymbol\theta\mid D)\propto p(D\mid\boldsymbol\theta)\,p(\boldsymbol\theta),
\qquad \boldsymbol\theta=(\tau,C,\omega,\phi).
$$

The frequency and phase can be correlated, so the joint posterior matters more than four separate error bars. `emcee` explores that target by producing correlated draws. A **normalizing flow** offers a different representation: it transforms an easy-to-sample normal vector into parameter draws from a flexible approximation $q_{\boldsymbol\eta}(\boldsymbol\theta)$. Its inverse and Jacobian, together with the known normal base density, let us evaluate $q_{\boldsymbol\eta}$ at a proposed point. This lets us draw approximate parameter values and evaluate a loss that compares the approximation with the physical posterior.

[`zuko`](https://zuko.readthedocs.io/stable/) supplies a ready-made flow implemented in PyTorch. The **machine learning** step is to adjust its parameters $\boldsymbol\eta$ using derivatives of the loss; the Ramsey model, observed counts, and prior still define the physics. We can optimize directly against the differentiable binomial likelihood for this **single dataset**, without training on `emcee` samples. Later we will compare the resulting marginals, correlations, and predicted Ramsey curves with an `emcee` reference.

For four parameters, `emcee` is already practical, so the flow is a method to evaluate critically rather than an assumed speed improvement. Training costs time and a variational flow can miss a posterior mode or understate tails. The comparison will tell us what accuracy this approximation achieves for the same data.

## Part 1: The same Ramsey measurements

At each selected delay $t_i$, we make $n_i=250$ independent binary measurements (shots) and record one excited-state count $k_i$. One row of `ramsey_data_df` is this **aggregate count at one delay**, not one individual shot. For this synthetic dataset we choose $N_{\mathrm{delay}}=31$ equally spaced delays from $0$ to $5\,\mu\mathrm{s}$. Thus the table has 31 rows and represents $31\times250=7750$ shots in total. **31 is simply the number of sampled delay settings** chosen for consistency with earlier lectures; the likelihood construction works with any number of rows. The 250 shots per delay are also a simulation choice, and $n_i$ could differ between rows in real data. The probability model is

$$
p_e(t\mid\tau,C,\omega,\phi)=\frac12\left[1+C e^{-t/\tau}\cos(\omega t+\phi)\right],
\qquad k_i\sim\operatorname{Binomial}(n_i,p_e(t_i)).
$$

Here $\tau$ is the decay time, $C$ is contrast, $\omega$ is angular frequency, and $\phi$ is phase. We reproduce the same seed, delays, shot totals, generating parameters, and counts used in Lectures 17–19. The random seed makes this particular simulated dataset reproducible. Generating values are shown only because this is a simulation; they do not define the prior or train the flow. The plotted point is the measured fraction $\hat p_i=k_i/n_i$; its vertical bar is the approximate binomial standard error $\sqrt{\hat p_i(1-\hat p_i)/n_i}$. These bars display shot noise, while inference below uses the **counts and shot totals directly** in the binomial likelihood.

In [ ]:
# NumPy and pandas retain the same tabular measurements as the earlier lectures.
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import emcee
from scipy.stats import binom

plt.rcParams.update({'figure.figsize': (8, 4.5), 'axes.grid': True,
                     'grid.alpha': 0.25, 'font.size': 11})
torch.set_default_dtype(torch.float64)  # Reduce roundoff in log probabilities.

# Return the physical excited-state probability for array-like delay times.
def ramsey_probability(t, contrast, tau, omega, phase):
    t = np.asarray(t)
    p = 0.5 * (1.0 + contrast * np.exp(-t / tau) * np.cos(omega * t + phase))
    return np.clip(p, 1e-12, 1.0 - 1e-12)

true_parameters = {'contrast': 0.78, 'tau': 2.40,
                   'omega': 2.0 * np.pi * 0.85, 'phase': 0.25}
# Choose the same number of delay settings as Lectures 17–19.
n_delay_points = 31
ramsey_data_df = pd.DataFrame({'time_us': np.linspace(0.0, 5.0, n_delay_points)})
# Each row summarizes 250 binary shots at its delay.
ramsey_data_df['shots'] = 250
data_rng = np.random.default_rng(69017)
ramsey_data_df['generating_probability'] = ramsey_probability(
    ramsey_data_df['time_us'], **true_parameters
)
ramsey_data_df['excited_counts'] = data_rng.binomial(
    ramsey_data_df['shots'], ramsey_data_df['generating_probability']
)
ramsey_data_df['measured_probability'] = (
    ramsey_data_df['excited_counts'] / ramsey_data_df['shots']
)
ramsey_data_df['binomial_se'] = np.sqrt(
    ramsey_data_df['measured_probability']
    * (1.0 - ramsey_data_df['measured_probability']) / ramsey_data_df['shots']
)

# Plot through the DataFrame; 500 plot positions make a smooth curve, not new data.
time_fine = np.linspace(ramsey_data_df['time_us'].min(),
                        ramsey_data_df['time_us'].max(), 500)
fig, ax = plt.subplots()
ax.errorbar(ramsey_data_df['time_us'], ramsey_data_df['measured_probability'],
            yerr=ramsey_data_df['binomial_se'], fmt='o', capsize=2,
            label='same synthetic counts as Lectures 17–19')
ax.plot(time_fine, ramsey_probability(time_fine, **true_parameters),
        lw=2, label='generating probability')
ax.set(xlabel=r'Delay $t$ ($\mu\mathrm{s}$)', ylabel='Excited-state fraction',
       ylim=(0, 1), title='The fixed Ramsey dataset')
ax.legend()
plt.show()

## Part 2: A reference posterior from `emcee`

Use the same four parameters and uniform prior bounds as Lecture 18's worked extension. These ranges are **assumptions for this teaching example**, not values measured from the 31 aggregate-count rows. In particular, the narrow $\omega$ range stands in for external frequency knowledge and avoids distant alias solutions; a real analysis would need to justify it independently. The generating value is used only as a visual reference. The uniform prior density is constant inside the box and zero outside; its log is an additive constant inside and $-\infty$ outside. Consequently, `log_posterior_emcee` checks the bounds and returns the log likelihood without that additive constant. At an allowed parameter point, its numerical value should match the PyTorch log likelihood in Part 3.

| Parameter | Uniform prior range | Units |
|---|---:|---|
| $\tau$ | $[0.4,6.0]$ | $\mu\mathrm{s}$ |
| $C$ | $[0,1]$ | dimensionless |
| $\omega$ | $2\pi[0.70,1.00]$ | $\mathrm{rad}/\mu\mathrm{s}$ |
| $\phi$ | $[-\pi,\pi]$ | rad |

We use `emcee` for a reference, **not** to supply training samples to the flow. The walker center and widths in the code choose algorithm starting positions; they are not additional prior information. The 48 walkers and warmup/production lengths are practical sampling settings, whose adequacy needs the diagnostics from Lecture 19. This reference has finite Monte Carlo error and is not an exact posterior.

In [ ]:
parameter_names = ['tau', 'contrast', 'omega', 'phase']
prior_bounds = {
    'tau': (0.4, 6.0), 'contrast': (0.0, 1.0),
    'omega': (2.0 * np.pi * 0.70, 2.0 * np.pi * 1.00),
    'phase': (-np.pi, np.pi),
}

# Evaluate the four-parameter target used by emcee; omit the constant log prior.
def log_posterior_emcee(parameters):
    if any(not prior_bounds[name][0] <= value <= prior_bounds[name][1]
           for name, value in zip(parameter_names, parameters)):
        return -np.inf
    tau, contrast, omega, phase = parameters
    probability = ramsey_probability(
        ramsey_data_df['time_us'].to_numpy(), contrast, tau, omega, phase
    )
    return float(np.sum(binom.logpmf(
        ramsey_data_df['excited_counts'].to_numpy(),
        ramsey_data_df['shots'].to_numpy(), probability
    )))

# Randomize starting positions near a plausible point; these settings do not set the prior.
nwalkers, warmup_steps, production_steps = 48, 800, 3000
start_rng = np.random.default_rng(2020)
initial_center = np.array([2.2, 0.70, 2.0 * np.pi * 0.82, 0.2])
initial_scales = np.array([0.15, 0.04, 0.08, 0.15])
initial_positions = initial_center + start_rng.normal(
    size=(nwalkers, len(parameter_names))
) * initial_scales
if not np.all(np.isfinite([log_posterior_emcee(p) for p in initial_positions])):
    raise ValueError('An initial walker is outside the prior')

# Discard warmup states, then retain a reference sample from the posterior.
np.random.seed(2021)
sampler = emcee.EnsembleSampler(nwalkers, len(parameter_names), log_posterior_emcee)
emcee_start = time.perf_counter()
state = sampler.run_mcmc(initial_positions, warmup_steps, progress=False)
sampler.reset()
sampler.run_mcmc(state, production_steps, progress=False)
emcee_seconds = time.perf_counter() - emcee_start
reference_df = pd.DataFrame(sampler.get_chain(flat=True), columns=parameter_names)
tau_int = sampler.get_autocorr_time(tol=0)
reference_check_df = pd.DataFrame({
    'parameter': parameter_names,
    'median': reference_df[parameter_names].median().to_numpy(),
    'autocorrelation_time_steps': tau_int,
    'approximate_ESS': nwalkers * production_steps / tau_int,
})
print('Mean acceptance fraction:', round(sampler.acceptance_fraction.mean(), 3))
print('Elapsed emcee time (s):', round(emcee_seconds, 1))
display(reference_check_df)

The table is a quick check, not a convergence guarantee. The effective sample size (ESS) divides the total retained walker draws by an estimated autocorrelation time to approximate how many independent draws carry comparable information. It is a numerical sampling diagnostic, not a count of new measurements. Use Lecture 19's traces and independent-chain checks if this reference looks suspicious. The flow will be judged against the posterior represented by these draws.

In [ ]:
# Inspect the two parameter pairs most relevant to uncertainty and identifiability.
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
plot_reference_df = reference_df.iloc[::30]
for ax, x_name, y_name in [
    (axes[0], 'tau', 'contrast'), (axes[1], 'omega', 'phase')
]:
    ax.scatter(plot_reference_df[x_name], plot_reference_df[y_name],
               s=5, alpha=0.18, label='emcee draws')
    ax.scatter(true_parameters[x_name], true_parameters[y_name],
               marker='*', color='gold', edgecolor='black', s=100,
               label='generating values')
    ax.set(xlabel=x_name, ylabel=y_name, title=f'{x_name} and {y_name}')
    ax.legend(fontsize=8)
plt.show()

## Part 3: Connect the flow to the Ramsey data through the likelihood

The observations $D=\{(t_i,n_i,k_i)\}_{i=1}^{N_{\mathrm{delay}}}$ are fixed. Here $N_{\mathrm{delay}}$ means the number of rows in `ramsey_data_df`, which is 31 for this synthetic dataset. For a proposed parameter vector $\boldsymbol\theta=(\tau,C,\omega,\phi)$, the Ramsey model predicts one excited-state probability $p_e(t_i;\boldsymbol\theta)$ at each delay. The binomial likelihood asks how probable the **observed counts** $k_i$ would be at those predicted probabilities. Because delays are independent in this model, their log probabilities add:

$$
\ell(\boldsymbol\theta)
=\log p(D\mid\boldsymbol\theta)
=\sum_{i=1}^{N_{\mathrm{delay}}}\log\!\left[\operatorname{Binomial}\bigl(k_i\mid n_i,p_e(t_i;\boldsymbol\theta)\bigr)\right].
$$

This is the **same likelihood** used by `emcee` in Part 2. The change is computational: in Part 4, the flow will draw many $\boldsymbol\theta$ values and call `log_likelihood_torch` to score each one. A higher log likelihood says that a proposal better explains these counts. The flow also evaluates its own log density $\log q_{\boldsymbol\eta}(\boldsymbol\theta)$; together, these quantities make the training loss. The density term helps the optimizer learn a **distribution** rather than placing every draw at one best-fit point. The prior supplies the allowed parameter region.

One training step will therefore use the following sequence:

1. Zuko draws parameter vectors $\boldsymbol\theta$ from its current approximation $q_{\boldsymbol\eta}$.
2. The Ramsey model calculates $p_e(t_i;\boldsymbol\theta)$ at every measured delay.
3. The binomial log likelihood $\ell(\boldsymbol\theta)$ scores the fixed observed counts.
4. The loss combines $\log q_{\boldsymbol\eta}(\boldsymbol\theta)$, $-\ell(\boldsymbol\theta)$, and the prior (whose log is constant inside the allowed range).
5. `loss.backward()` and Adam update the flow parameters $\boldsymbol\eta$.

PyTorch must calculate the model and log likelihood using **tensor operations** so `loss.backward()` can carry derivatives through this path. The `clamp` in the code guards against taking $\log 0$ if a predicted probability reaches an endpoint; it is a numerical safeguard, not part of the physical model. Passing a proposed tensor into the earlier NumPy/SciPy likelihood would break that derivative path. We keep measurements in `ramsey_data_df` for inspection and plots, then copy its fixed columns into tensors once for repeated likelihood evaluations.

| Input to `log_likelihood_torch` | Model probabilities | Output after summing all measured delays |
|---|---|---|
| One $\boldsymbol\theta$: shape `(4,)` | shape `(N_delay,)` = `(31,)` here | one log likelihood (scalar) |
| $S$ proposals: shape `(S, 4)` | shape `(S, N_delay)` = `(S, 31)` here | shape `(S,)`, one score per proposal |

In [ ]:
# Copy DataFrame columns into tensors for vectorized, differentiable evaluations.
time_tensor = torch.tensor(ramsey_data_df['time_us'].to_numpy())
shots_tensor = torch.tensor(ramsey_data_df['shots'].to_numpy(), dtype=torch.float64)
counts_tensor = torch.tensor(ramsey_data_df['excited_counts'].to_numpy(), dtype=torch.float64)

# Evaluate p_e for one parameter vector or a batch, broadcasting across delays.
def ramsey_probability_torch(theta_batch):
    tau, contrast, omega, phase = theta_batch.unbind(dim=-1)
    # Add a delay axis so a batch of proposals scores every measured time.
    tau = tau.unsqueeze(-1)
    contrast = contrast.unsqueeze(-1)
    omega = omega.unsqueeze(-1)
    phase = phase.unsqueeze(-1)
    probability = 0.5 * (1.0 + contrast
        * torch.exp(-time_tensor / tau)
        * torch.cos(omega * time_tensor + phase))
    # Avoid exact 0 or 1, where a binomial log probability may diverge.
    return probability.clamp(1e-12, 1.0 - 1e-12)

### Worked bridge: score one observed count

Before summing all delays, evaluate the binomial log probability for the **first DataFrame row** at one plausible parameter vector. The known shot count $n_1$, observed excited count $k_1$, and model probability $p_e(t_1;\boldsymbol\theta)$ are the three arguments needed for this row. The parameter vector is a test point inside the prior, not the generating value or a fitted estimate. The next activity repeats this calculation for every DataFrame row and sums the answers.

In [ ]:
# Evaluate at a test point inside the prior, chosen for illustration rather than fit quality.
check_theta = torch.tensor([2.2, 0.70, 2.0 * np.pi * 0.82, 0.2])
check_probabilities = ramsey_probability_torch(check_theta)
print('Probability shape for one parameter vector:', tuple(check_probabilities.shape))

# Show the observed count that the first binomial distribution will score.
display(ramsey_data_df[['time_us', 'shots', 'excited_counts']].iloc[[0]])
first_delay_distribution = torch.distributions.Binomial(
    total_count=shots_tensor[0], probs=check_probabilities[0]
)
first_log_probability = first_delay_distribution.log_prob(counts_tensor[0])
print('Model p_e at first delay:', round(check_probabilities[0].item(), 4))
print('Log probability of first observed count:', round(first_log_probability.item(), 3))

### In-class coding activity 1: complete the full-data likelihood

Generalize the one-row calculation above. `shots_tensor` and `counts_tensor` contain one entry per measured delay; `model_probability` already contains the predictions for one parameter point or a batch.

1. In `student_log_likelihood_torch`, construct `torch.distributions.Binomial(total_count=shots_tensor, probs=model_probability)`.
2. Call `.log_prob(counts_tensor)` and sum over the delay axis with `.sum(dim=-1)`. The function should return a scalar for one $\boldsymbol\theta$ or one value per row in a parameter batch.
3. When ready, try the commented checks. Compare your result with `log_posterior_emcee(check_theta.numpy())`; its omitted constant uniform prior means the log-likelihood values should agree at an allowed point.

This exercise uses its **own function name**. The completed implementation in the next cell supplies `log_likelihood_torch` for the rest of the lecture, so you can continue even if your TODOs are unfinished.

**Discuss before coding:** Why do we add log probabilities across the measured delays rather than multiplying many small probabilities?

In [ ]:
def student_log_likelihood_torch(theta_batch):
    """Your differentiable binomial log likelihood for each parameter draw."""
    model_probability = ramsey_probability_torch(theta_batch)
    # TODO: Create a Binomial distribution with shots_tensor and model_probability.
    count_distribution = ...
    # TODO: Evaluate observed counts and sum log probabilities over delay points.
    log_likelihood = ...
    return log_likelihood

# After completing the TODOs, uncomment these checks in your own cell:
# print('Your log likelihood:', student_log_likelihood_torch(check_theta).item())
# print('SciPy log likelihood:', log_posterior_emcee(check_theta.numpy()))
# print('Batch shape:', student_log_likelihood_torch(
#     torch.stack([check_theta, check_theta])).shape)

### Continue with the completed likelihood

This implementation uses the same binomial calculation as the activity. The numerical comparison with SciPy and the batch-shape check verify the function used in Parts 4 and 5. The training loop calls `log_likelihood_torch`; it does not call `student_log_likelihood_torch`.

In [ ]:
def log_likelihood_torch(theta_batch):
    """Return one differentiable log likelihood for each parameter draw."""
    model_probability = ramsey_probability_torch(theta_batch)
    # Compare every observed count with its predicted binomial distribution.
    count_distribution = torch.distributions.Binomial(
        total_count=shots_tensor, probs=model_probability
    )
    # Sum independent delay-point contributions, preserving any batch axis.
    return count_distribution.log_prob(counts_tensor).sum(dim=-1)

# Both methods use the same fixed counts and the same physical model.
scipy_log_likelihood = log_posterior_emcee(check_theta.numpy())
torch_log_likelihood = log_likelihood_torch(check_theta).item()
print('SciPy:', scipy_log_likelihood)
print('PyTorch:', torch_log_likelihood)
print('Absolute difference:', abs(scipy_log_likelihood - torch_log_likelihood))
assert np.isclose(scipy_log_likelihood, torch_log_likelihood, atol=1e-6)

# A flow sends batches of proposals, so check that two proposals give two values.
second_theta = check_theta + torch.tensor([0.1, 0.0, 0.0, 0.0])
check_batch = torch.stack([check_theta, second_theta])
print('Two-proposal log-likelihood shape:', tuple(log_likelihood_torch(check_batch).shape))
assert log_likelihood_torch(check_batch).shape == (2,)

### Why this function must remain differentiable

The toy example before Part 1 differentiated a loss with respect to two adjustable numbers. We can now differentiate the **actual Ramsey log likelihood** with respect to $\tau,C,\omega,\phi$. These four derivatives tell us how the score changes locally when the proposed physics parameters move. During flow training, PyTorch applies the chain rule again, through the proposal $\boldsymbol\theta$ back to Zuko's trainable flow parameters $\boldsymbol\eta$. The observed counts are fixed; they are not trainable values.

In [ ]:
# Check that the physical log likelihood has a finite four-component gradient.
theta_for_gradient = check_theta.clone().detach().requires_grad_(True)
log_likelihood_torch(theta_for_gradient).backward()
gradient_df = pd.DataFrame({
    'parameter': parameter_names,
    'd_log_likelihood_d_parameter': theta_for_gradient.grad.detach().numpy(),
})
display(gradient_df)
assert np.isfinite(gradient_df['d_log_likelihood_d_parameter']).all()

The same function can score many candidate frequencies in one call. The 100 frequencies in the next code cell are **trial parameter values for a smooth diagnostic plot**, not 100 additional measurements. The completed code below makes a **conditional likelihood slice**: hold $\tau$, $C$, and $\phi$ at `check_theta`, vary only $\omega$, and subtract the largest log likelihood. This is not a profile likelihood because the other three parameters are not reoptimized at each frequency. It makes the frequency dependence visible before we optimize a four-dimensional density.

In [ ]:
# Scan 100 trial frequencies for a smooth diagnostic plot; the observed data stay fixed.
frequency_grid = torch.linspace(prior_bounds['omega'][0],
                                prior_bounds['omega'][1], 100)
candidate_parameters = check_theta.repeat(len(frequency_grid), 1)
candidate_parameters[:, 2] = frequency_grid
# Score all candidate frequencies in one batch and shift the maximum to zero.
frequency_log_likelihood = log_likelihood_torch(candidate_parameters)
delta_log_likelihood = frequency_log_likelihood - frequency_log_likelihood.max()

fig, ax = plt.subplots()
ax.plot(frequency_grid.numpy(), delta_log_likelihood.detach().numpy())
ax.axvline(true_parameters['omega'], color='black', ls='--',
           label='generating frequency')
ax.set(xlabel=r'$\omega$ ($\mathrm{rad}/\mu\mathrm{s}$)',
       ylabel=r'$\log L(\omega)-\max\log L$',
       title=r'Conditional likelihood slice at fixed $\tau$, $C$, $\phi$')
ax.legend()
plt.show()

## Part 4: Direct variational inference with a production flow

The completed `log_likelihood_torch` from Part 3 scores the fixed Ramsey counts at any proposed $\boldsymbol\theta$. Now Zuko will produce those proposals. A normalizing flow starts with a simple latent distribution and applies trainable, invertible transformations to produce a density $q_{\boldsymbol\eta}$. Zuko's neural spline flow (`NSF`) supplies flexible invertible nonlinear transformations, their network parameters, and the change-of-variables density calculation. Here a **spline** is a smooth, piecewise transformation whose shape the network learns. In the code, `features=4` matches the four inferred physics parameters; `transforms=2` and `hidden_features=(32, 32)` choose model capacity and computing cost, not properties of the Ramsey experiment. The flow learns a posterior **for this fixed dataset** by calling `log_likelihood_torch` inside its loss; it is not trained on the `emcee` draws.

The physical priors have finite bounds. We let Zuko produce an unconstrained vector $\mathbf u\in\mathbb R^4$, then map each component into its prior interval:

$$
\theta_j=a_j+(b_j-a_j)\,\operatorname{sigmoid}(u_j),
\qquad \operatorname{sigmoid}(u)=\frac{1}{1+e^{-u}}.
$$

Here $a_j,b_j$ are the bounds in Part 2. This map prevents invalid $\tau$ or $C$ values and keeps frequency and phase within their declared priors. The density in physical coordinates must account for the map's Jacobian: $\log q_{\boldsymbol\eta}(\boldsymbol\theta)=\log q_{\boldsymbol\eta}(\mathbf u)-\log|\det(\partial\boldsymbol\theta/\partial\mathbf u)|$. Zuko computes the **flow's** Jacobian; the short function below computes only the known sigmoid-to-prior-box correction.

We minimize a Monte Carlo estimate of

$$
\mathcal L(\boldsymbol\eta)=
\operatorname{average}_{\boldsymbol\theta\sim q_{\boldsymbol\eta}}
\left[\log q_{\boldsymbol\eta}(\boldsymbol\theta)
-\log p(D\mid\boldsymbol\theta)-\log p(\boldsymbol\theta)\right].
$$

This objective is called the **negative evidence lower bound** (negative ELBO). The evidence $p(D)$ is the probability of the fixed dataset averaged over the prior; its logarithm is constant with respect to the flow parameters $\boldsymbol\eta$. Minimizing the displayed loss therefore moves $q_{\boldsymbol\eta}$ toward the posterior without calculating $p(D)$. The sigmoid map keeps all draws inside the uniform prior box, where $\log p(\boldsymbol\theta)$ is constant. That is why the code uses only `log_q_draws - log_likelihood_draws`: omitting the prior constant changes the loss value but not its minimum. PyTorch's `backward()` obtains the loss derivatives and Adam updates Zuko's trainable parameters. The Kullback–Leibler (KL) divergence measures a mismatch between densities. The part of this objective that depends on the flow is $\mathrm{KL}(q_{\boldsymbol{\eta}}\,\|\,p(\boldsymbol{\theta}\mid D))$, called **reverse KL** because the flow density comes first. It averages over draws from $q_{\boldsymbol\eta}$, so a posterior mode the flow rarely visits may be missed. A good-looking loss curve is not enough to validate the posterior.

In [ ]:
# Define the physical prior box in the same parameter order as the likelihood.
prior_lower = torch.tensor([prior_bounds[name][0] for name in parameter_names])
prior_upper = torch.tensor([prior_bounds[name][1] for name in parameter_names])
prior_width = prior_upper - prior_lower

# features is the four-parameter dimension; the other settings tune flow flexibility.
torch.manual_seed(2026)
flow = zuko.flows.NSF(features=len(parameter_names), transforms=2,
                      hidden_features=(32, 32)).to(dtype=torch.float64)

def draw_bounded_flow(batch_size):
    """Draw physical parameters and return their normalized log density."""
    # Zuko accounts for all Jacobians inside its neural spline flow.
    unconstrained, log_q_unconstrained = flow().rsample_and_log_prob((batch_size,))
    fraction = torch.sigmoid(unconstrained)
    theta = prior_lower + prior_width * fraction
    # Correct the density for the sigmoid map into the prior box.
    log_box_jacobian = (
        torch.log(prior_width)
        + torch.nn.functional.logsigmoid(unconstrained)
        + torch.nn.functional.logsigmoid(-unconstrained)
    ).sum(dim=-1)
    return theta, log_q_unconstrained - log_box_jacobian

# Check the sample and density shapes before optimizing.
theta_test, log_q_test = draw_bounded_flow(8)
print('Parameter sample shape:', tuple(theta_test.shape))
print('Log density shape:', tuple(log_q_test.shape))
assert theta_test.shape == (8, 4) and log_q_test.shape == (8,)

### Worked training loop

Each step draws fresh parameter vectors from the current flow, scores them with the Ramsey likelihood and the flow density, and changes the flow to lower the average loss. The model probability is evaluated for the **observed** counts every time; the generating parameters are never used in the loss. The 128 parameter draws per step and 500 optimizer steps are illustrative training settings; they are unrelated to the number of measured delays or shots. We time the complete training loop to compare with `emcee` on the same computer. Runtime alone does not establish which posterior is more accurate.

In [ ]:
optimizer = torch.optim.Adam(flow.parameters(), lr=2e-3)
training_loss = []
flow_start = time.perf_counter()

for step in range(1, 501):
    optimizer.zero_grad()
    # Draw 128 candidate parameter vectors for this optimizer update.
    theta_draws, log_q_draws = draw_bounded_flow(128)
    # Score the fixed counts; gradients pass through theta_draws to the flow.
    log_likelihood_draws = log_likelihood_torch(theta_draws)
    loss = (log_q_draws - log_likelihood_draws).mean()
    if not torch.isfinite(loss):
        raise RuntimeError('Non-finite training loss; inspect the likelihood and flow')
    loss.backward()
    torch.nn.utils.clip_grad_norm_(flow.parameters(), max_norm=10.0)
    optimizer.step()
    training_loss.append(loss.detach().item())
    if step % 100 == 0:
        print(f'Step {step:3d}: loss = {training_loss[-1]:.2f}')

flow_seconds = time.perf_counter() - flow_start
print('Elapsed flow training time (s):', round(flow_seconds, 1))

# Inspect whether optimization settled; fluctuations reflect fresh Monte Carlo draws.
fig, ax = plt.subplots()
ax.plot(np.arange(1, len(training_loss) + 1), training_loss, lw=1)
ax.set(xlabel='Optimization step', ylabel='Monte Carlo loss',
       title='Direct variational flow training')
plt.show()

## Part 5: Compare with the posterior reference

The training objective used neither `reference_df` nor its covariance. We can now use those `emcee` draws as an independent comparison; the quick checks in Part 2 do not replace the full diagnostics from Lecture 19. First compare the marginal distributions and two correlated parameter pairs; then compare the predictions $p_e(t)$, which are the experiment's observable. A posterior approximation can look close in one dimension while distorting a correlation or a predictive band.

In [ ]:
# Draw 12,000 approximate parameter vectors for comparison plots, not new data.
flow.eval()
with torch.no_grad():
    final_theta, _ = draw_bounded_flow(12000)
flow_df = pd.DataFrame(final_theta.cpu().numpy(), columns=parameter_names)

# Show every marginal distribution against the emcee reference.
fig, axes = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
for ax, name in zip(axes.flat, parameter_names):
    ax.hist(reference_df[name], bins=50, density=True, histtype='step',
            lw=2, label='emcee')
    ax.hist(flow_df[name], bins=50, density=True, histtype='step',
            lw=2, label='PyTorch + Zuko')
    ax.axvline(true_parameters[name], color='black', ls='--',
               lw=1, label='generating value')
    ax.set(xlabel=name, ylabel='Density')
    ax.legend(fontsize=8)
plt.show()

# Paired plots reveal whether correlations survive the approximation.
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for ax, x_name, y_name in [
    (axes[0], 'tau', 'contrast'), (axes[1], 'omega', 'phase')
]:
    ax.scatter(reference_df[x_name].iloc[::30],
               reference_df[y_name].iloc[::30],
               s=5, alpha=0.13, label='emcee')
    ax.scatter(flow_df[x_name].iloc[::4], flow_df[y_name].iloc[::4],
               s=5, alpha=0.13, label='flow')
    ax.set(xlabel=x_name, ylabel=y_name, title=f'{x_name} and {y_name}')
    ax.legend(fontsize=8)
plt.show()

### In-class coding activity 2: quantify agreement and check Ramsey predictions

Work with `reference_df` and `flow_df` in the **student** functions below. Compute each parameter's median and central 68% interval (the 16th to 84th percentiles), subtract their Pearson correlation matrices, and propagate complete parameter rows into a 68% model-probability band. The band reflects parameter uncertainty, not the shot noise of a future measurement.

Try the commented checks after completing your TODOs. The following completed cell independently constructs the summary table, correlation comparison, and probability-band function for the final plot. Your work remains available for comparison, and the lecture continues if a TODO is unfinished.

**Discussion:** If a broader frequency prior produced a second posterior peak but the flow showed only one, would agreement of the median alone make the flow trustworthy?

In [ ]:
# Build your own interval summary from a labeled posterior-sample DataFrame.
def student_interval_summary(samples_df):
    """Return medians and central 68% intervals for all Ramsey parameters."""
    # TODO: Select parameter_names and calculate the 0.16, 0.50, 0.84 quantiles.
    quantiles = ...
    return pd.DataFrame({
        'parameter': parameter_names,
        'lower_16': quantiles.loc[0.16, parameter_names].to_numpy(),
        'median': quantiles.loc[0.50, parameter_names].to_numpy(),
        'upper_84': quantiles.loc[0.84, parameter_names].to_numpy(),
    })

# Compare correlations for all pairs of physics parameters.
def student_correlation_difference(reference_samples_df, flow_samples_df):
    reference_correlation_df = reference_samples_df[parameter_names].corr()
    flow_correlation_df = flow_samples_df[parameter_names].corr()
    # TODO: Subtract the emcee correlation matrix from the flow matrix.
    correlation_difference_df = ...
    return correlation_difference_df

# Propagate complete correlated parameter rows to probability curves.
def student_probability_band(samples_df, times, n_curves=500):
    selected_indices = np.linspace(0, len(samples_df) - 1, n_curves, dtype=int)
    selected_df = samples_df[parameter_names].iloc[selected_indices]
    # TODO: Broadcast DataFrame parameter columns against the time grid.
    curves = ...
    # TODO: Return the 16th, 50th, 84th percentile curves over sample axis 0.
    return ...

# After completing the TODOs, try in your own cell:
# display(student_interval_summary(reference_df))
# display(student_correlation_difference(reference_df, flow_df))
# print('Probability-band shapes:', [array.shape for array in
#       student_probability_band(flow_df, time_fine)])

### Continue with completed posterior comparisons

The summary and band below use complete implementations. The final Ramsey plot calls `probability_band`, so it runs independently of `student_probability_band` and the other activity functions. Compare these results with your own calculations after the activity.

In [ ]:
# Calculate central 68% parameter intervals from labeled posterior draws.
def interval_summary(samples_df):
    """Return medians and central 68% intervals for all Ramsey parameters."""
    quantiles = samples_df[parameter_names].quantile([0.16, 0.50, 0.84])
    return pd.DataFrame({
        'parameter': parameter_names,
        'lower_16': quantiles.loc[0.16, parameter_names].to_numpy(),
        'median': quantiles.loc[0.50, parameter_names].to_numpy(),
        'upper_84': quantiles.loc[0.84, parameter_names].to_numpy(),
    })

reference_summary_df = interval_summary(reference_df)
flow_summary_df = interval_summary(flow_df)
comparison_df = reference_summary_df.merge(
    flow_summary_df, on='parameter', suffixes=('_emcee', '_flow')
)
display(comparison_df)

# Pearson correlations are dimensionless; compare all parameter pairs.
reference_correlation_df = reference_df[parameter_names].corr()
flow_correlation_df = flow_df[parameter_names].corr()
correlation_difference_df = flow_correlation_df - reference_correlation_df
display(correlation_difference_df)

# Use joint parameter rows to preserve correlations in each model prediction.
def probability_band(samples_df, times, n_curves=500):
    selected_indices = np.linspace(0, len(samples_df) - 1, n_curves, dtype=int)
    selected_df = samples_df[parameter_names].iloc[selected_indices]
    time_values = np.asarray(times)[None, :]
    curves = ramsey_probability(
        time_values,
        contrast=selected_df['contrast'].to_numpy()[:, None],
        tau=selected_df['tau'].to_numpy()[:, None],
        omega=selected_df['omega'].to_numpy()[:, None],
        phase=selected_df['phase'].to_numpy()[:, None],
    )
    return np.quantile(curves, [0.16, 0.50, 0.84], axis=0)

In [ ]:
# Plot the measured counts and the two posterior model-probability bands.
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.errorbar(ramsey_data_df['time_us'], ramsey_data_df['measured_probability'],
            yerr=ramsey_data_df['binomial_se'], fmt='o', ms=3,
            color='black', capsize=2, label='measured fractions')
for samples_df, color, label in [
    (reference_df, 'C0', 'emcee reference'),
    (flow_df, 'C1', 'PyTorch flow'),
]:
    lower, median, upper = probability_band(samples_df, time_fine)
    ax.plot(time_fine, median, color=color, lw=1.5, label=label)
    ax.fill_between(time_fine, lower, upper, color=color, alpha=0.16)
ax.set(xlabel=r'Delay $t$ ($\mu\mathrm{s}$)', ylabel=r'$p_e(t)$', ylim=(0, 1),
       title='Posterior predictions for the same Ramsey data')
ax.legend()
plt.show()

### What does this comparison establish?

For the fixed seeds and settings here, the **central 68% interval widths agree within a few percent**. The flow minus `emcee` median is approximately $0.0001\,\mu\mathrm{s}$ for $\tau$, $0.0007$ for $C$, $0.009\,\mathrm{rad}/\mu\mathrm{s}$ for $\omega$, and $0.017\,\mathrm{rad}$ for $\phi$. The frequency and phase shifts are roughly one-quarter to one-third of their marginal posterior standard deviations. The median predicted probabilities differ by less than about $0.01$ over the plotted time grid. Check the table and plot from your own run: small changes in sampling and optimization can change these numbers. Correlated parameters can shift while still producing similar probability curves.

The printed `flow_seconds` times **training**, and `emcee_seconds` times **MCMC sampling**. In one local run they were about 6 and 15 seconds, respectively. This shows that this training loop ran faster on that machine; it is not a comparison at matched posterior accuracy and excludes setup, validation, and repeated training runs. Flow draws are cheap *after* training, but this flow must be trained again for a new dataset. A smaller flow interval would not represent more information in the Ramsey measurements: both methods target the same posterior.

**For this single four-parameter dataset, a well-diagnosed `emcee` run is the simpler default.** Keeping both methods in this lecture is useful as a cross-check and as a demonstration of a learned posterior approximation; their results should not be averaged or combined as independent measurements. A flow becomes compelling if validated against a trustworthy reference and its faster repeated draws or density evaluations save enough time to offset training and checking. This particular run does not establish that the flow is more accurate, or that its modest runtime gain is useful for this analysis.

Reverse-KL training may concentrate on one mode when a broader frequency prior admits several alias solutions. A low training loss can coexist with missing modes or underestimated tails. For a serious analysis, try multiple flow initializations, inspect the frequency profile and pair plots, use posterior predictive checks, and compare with independent MCMC chains. If those checks fail, do not quote the flow intervals as physical uncertainty. This is the principal limitation illustrated by the [Zuko reverse-KL tutorial](https://zuko.readthedocs.io/stable/tutorials/reverse_kl.html).


## Part 6: Final-project planning

For your final project, use this class period to choose a tractable physics question. Your method could involve fitting, ODE/PDE solutions, eigensystems, FFTs, Monte Carlo, MCMC, or an ML-assisted extension.

1. What observable or prediction will you study?
2. What is the smallest complete baseline calculation?
3. Which data, initial/boundary conditions, priors, and approximations are required?
4. What benchmark or diagnostic will show that the answer is trustworthy?
5. Which uncertainty comes from measurement or physics, and which from numerical approximation?
6. What script, notebook, figure, and README will make the result reproducible?

Use the [final-project template](../projects/project-template.md) and [rubric](../projects/rubrics.md) to check feasibility.

## Conclusion

- We kept the same Ramsey counts and likelihood while inferring four parameters rather than two.
- PyTorch evaluated the likelihood and its derivatives; Zuko supplied the trainable normalizing flow and its density.
- Direct variational training used the fixed data and prior, without using `emcee` draws as training data.
- The independent `emcee` posterior, pair plots, interval table, and Ramsey probability bands tested the approximation.
- For this four-parameter example, the flow gave similar predictions but did not establish better accuracy; `emcee` remains the simpler default. Training cost, validation, and missed modes determine whether a flow helps on a harder problem.

## References and further reading

- [PyTorch: tensors, autograd, and optimization](https://docs.pytorch.org/tutorials/beginner/basics/intro.html) — an accessible introduction to the operations used here.
- [PyTorch probability distributions](https://docs.pytorch.org/docs/stable/distributions.html) — documentation for `Binomial.log_prob`.
- [Zuko: normalizing flows in PyTorch](https://zuko.readthedocs.io/stable/) and [training from an unnormalized target](https://zuko.readthedocs.io/stable/tutorials/reverse_kl.html) — the production library and direct variational objective used in Part 4.
- [Rezende and Mohamed, *Variational Inference with Normalizing Flows*](https://arxiv.org/abs/1505.05770) — foundational open paper.
- [`emcee` quickstart](https://emcee.readthedocs.io/en/stable/tutorials/quickstart/) — the sampling baseline from Lectures 18 and 19.
- [Unit 4 course resources](../resources/unit-4-monte-carlo-inference.md).